In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from scipy.special import softmax

from dicom_report import generate_report
from dicom_report.utils.enums import Diagnosis

In [2]:
def image_to_case_id(image_path: str) -> str:
    return '_'.join(Path(image_path).name.split('_')[:2])


def get_prediction(risk_score: float, thresholds: tuple[float, float]) -> str:
    if risk_score < thresholds[0]:
        return Diagnosis.BENIGN
    if risk_score > thresholds[1]:
        return Diagnosis.MALIGNANT
    return Diagnosis.INCONCLUSIVE


def get_predictions(model_dir: Path, thresholds: tuple[float, float]) -> pd.DataFrame:
    df = pd.read_csv(model_dir / "DeiT_10.csv", index_col=0)
    df.loc[:, list(map(str, range(10)))] = softmax(df, axis=1)
    df = df[list(map(str, range(5, 10)))].sum(axis=1).to_frame('model')
    df = df.groupby(image_to_case_id).mean()
    df['diagnosis'] = df['model'].apply(lambda x: get_prediction(x, thresholds))
    return df

In [3]:
THRESHOLDS = (0.4, 0.65)
OMLC_WEBSITE_DATA_DIR = Path("/Users/filip.christiansen.2/src/omlc-website/data")
MODEL_DIR = Path("/Users/filip.christiansen.2/src/ovarian-project-analysis/data/model/logits")

In [4]:
omlc_website_cases_dir = OMLC_WEBSITE_DATA_DIR / "images"
reports_dir = OMLC_WEBSITE_DATA_DIR / "reports"

In [5]:
predictions_df = get_predictions(MODEL_DIR, THRESHOLDS)

In [6]:
case_dirs = sorted([d for d in omlc_website_cases_dir.iterdir() if d.is_dir()])

In [7]:
for case_dir in case_dirs:
    case_id = case_dir.name
    image_paths = sorted([d for d in case_dir.iterdir() if d.is_file() and d.suffix == ".jpg"])
    images = [Image.open(p) for p in image_paths]
    diagnosis = predictions_df.loc[case_id, 'diagnosis']
    report = generate_report(images, diagnosis=diagnosis)
    report.save(reports_dir / f'{case_id}.png')